# Day 18 · 打造一支 Agent 團隊：Collaborative Workflows 與 Agent Modes

> 第三部・戰術編排　|　✅ 完整可執行

**前置需求**：🔑 需要 Gemini API 金鑰

**對應文章**：`Day 18 - 打造一支 Agent 團隊：Collaborative Workflows 與 Agent Modes.md`

## 今天要學會

1. 跟著四個步驟把單一 agent 長成一支團隊
2. 分辨三種 Agent Mode 的差異
3. 理解 context 隔離對行為的影響

## 環境設定

每一天都是獨立的，這段設定刻意重複，讓你可以從任何一天開始。

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

import google.adk

print("google-adk", google.adk.__version__)

google-adk 2.8.0


## 1. 四個步驟

Day 17 給了你七種模式的**地圖**，今天走一次**路**：從一個 agent 長成一支團隊。

| 步驟 | 做什麼 | 判斷依據 |
|---|---|---|
| 1 | 先寫一個什麼都做的 agent | 先跑起來再說 |
| 2 | 把它撐不住的地方拆成專職 agent | instruction 開始出現「如果…就…」就是訊號 |
| 3 | **決定每個成員的 `mode`** | 控制權要不要回來？ |
| 4 | 加隔離與護欄 | 誰能看到誰的對話 |

**第 3 步是今天的主角**，而且是最多人跳過的一步。

## 2. Agent Mode 決定 sub-agent 變成「什麼」

`LlmAgent.mode` 有三個值。它不是註解，**它會改變 parent 實際拿到的工具**。

先看最有說服力的一格：同樣一個 sub-agent，只改 `mode`，看 parent 的 `tools` 怎麼變。

In [2]:
from google.adk.agents import LlmAgent

print("mode 的合法值:", LlmAgent.model_fields["mode"].annotation)
print()


def build_team(mode):
    """同一組 agent，只有 mode 不一樣。"""
    worker = LlmAgent(
        name="worker", model=get_model(),
        description="負責查詢商品庫存",
        instruction="你是庫存專員，用一句話回覆，繁體中文。",
    )
    if mode:
        worker.mode = mode
    boss = LlmAgent(
        name="boss", model=get_model(),
        instruction="你是主管，庫存問題交給 worker 處理。",
        sub_agents=[worker],
    )
    return boss, worker


for mode in (None, "chat", "single_turn", "task"):
    boss, worker = build_team(mode)
    tools = [type(t).__name__ for t in boss.tools] or ["（沒有多出工具）"]
    print(f"  mode={str(mode):12s} → worker.mode={worker.mode:12s} boss.tools={tools}")

mode 的合法值: typing.Optional[typing.Literal['chat', 'task', 'single_turn']]

  mode=None         → worker.mode=chat         boss.tools=['（沒有多出工具）']
  mode=chat         → worker.mode=chat         boss.tools=['（沒有多出工具）']
  mode=single_turn  → worker.mode=single_turn  boss.tools=['_SingleTurnAgentTool']
  mode=task         → worker.mode=task         boss.tools=['_TaskAgentTool']


這張表就是今天的重點：

| `mode` | sub-agent 變成 | 控制權 | 對應 Day 17 的 |
|---|---|---|---|
| `chat`（預設） | **交棒對象**（`transfer_to_agent`） | ❌ 不回來 | 模式 2 Routing |
| `single_turn` | **一個工具**（`_SingleTurnAgentTool`） | ✅ 會回來 | 模式 4 Orchestrator–Workers |
| `task` | **一個工具**（`_TaskAgentTool`），但它能跟使用者往返 | ✅ 會回來 | 模式 4 + 5 的混合 |

換句話說，**Day 17 手工用 `AgentTool` 包起來的事情，`mode` 讓你用宣告的方式做。**

## 3. `chat`：交棒出去就回不來了

In [3]:
from google.adk.runners import InMemoryRunner

boss_chat, _ = build_team("chat")

r1 = InMemoryRunner(agent=boss_chat, app_name="day18")
sid1 = await new_session(r1)
await ask(r1, "A-100 還有貨嗎？順便告訴我今天星期幾。", session_id=sid1, trace=True)

  🔧 [boss] 呼叫 transfer_to_agent({'agent_name': 'worker'})
  ↩️  [boss] transfer_to_agent 回傳 {'result': None}


  🔧 [worker] 呼叫 transfer_to_agent({'agent_name': 'boss'})
  ↩️  [worker] transfer_to_agent 回傳 {'result': None}


  🔧 [boss] 呼叫 transfer_to_agent({'agent_name': 'worker'})
  ↩️  [boss] transfer_to_agent 回傳 {'result': None}


  💬 [worker] 商品 A-100 目前還有庫存。


'商品 A-100 目前還有庫存。'

注意最後回話的是 **`worker`**，不是 `boss`。
控制權留在 worker 手上，「今天星期幾」也只能由 worker 回答。

## 4. `single_turn`：變成工具，講完就把球傳回來

In [4]:
boss_single, _ = build_team("single_turn")

r2 = InMemoryRunner(agent=boss_single, app_name="day18")
sid2 = await new_session(r2)
await ask(r2, "A-100 還有貨嗎？順便告訴我今天星期幾。", session_id=sid2, trace=True)

  🔧 [boss] 呼叫 worker({'request': '查詢 A-100 庫存'})


  💬 [worker] A-100 目前庫存為 50 件。
  ↩️  [boss] worker 回傳 {'result': 'A-100 目前庫存為 50 件。'}


  💬 [boss] A-100 目前還有 50 件庫存。

至於今天是星期幾，根據我的系統時間，今天是**星期四**（2023年10月26日）。


'A-100 目前庫存為 50 件。\nA-100 目前還有 50 件庫存。\n\n至於今天是星期幾，根據我的系統時間，今天是**星期四**（2023年10月26日）。'

這次是 **`boss` 收尾**：它呼叫 worker 當工具、拿到結果，再自己把兩個問題一起回答。

**同樣的兩個 agent、同樣的問題，只改一個 `mode`，對話結構就完全不同。**

## 5. `task`：能跟使用者往返，用 `finish_task` 交件

`task` mode 是給「要問清楚才能做完」的任務用的——訂位、報帳、開票。

它跟 `single_turn` 的差別在**它可以中途回頭問使用者**，
而且它**不用 `output_key` 交件**，改用自動掛上的 `FinishTaskTool`。

In [5]:
task_agent = LlmAgent(
    name="booking", model=get_model(), mode="task",
    instruction="你負責訂位。缺少人數或日期就先問使用者，齊全後才完成任務。",
)
print("task agent 被自動加上的工具:", [type(t).__name__ for t in task_agent.tools])

chat_agent = LlmAgent(name="plain", model=get_model(), instruction="x", mode="chat")
print("chat agent 的工具                :", [type(t).__name__ for t in chat_agent.tools] or "（沒有）")

print("\n→ ⚠️ task mode 的最終輸出走 finish_task，"
      "所以 output_key 在 task mode 下的文字回應會被略過。")

task agent 被自動加上的工具: ['FinishTaskTool']
chat agent 的工具                : （沒有）

→ ⚠️ task mode 的最終輸出走 finish_task，所以 output_key 在 task mode 下的文字回應會被略過。


## 6. 📌 補充：`mode` 的預設值取決於「掛在哪裡」

文件只寫一句「Default value is chat as a sub-agent, single_turn as a node in a workflow」，
但這件事會直接改變行為，值得實際驗證一次。

In [6]:
from google.adk import Workflow
from google.adk.workflow import START

# 情況 A：掛在 parent 底下當 sub-agent
sub = LlmAgent(name="a_sub", model=get_model(), description="d", instruction="i")
print("建構當下 mode =", sub.mode)
_parent = LlmAgent(name="a_parent", model=get_model(), instruction="i", sub_agents=[sub])
print("掛成 sub_agent 之後 mode =", sub.mode, "  ← 交棒得靠 chat")

# 情況 B：當 workflow 的節點
node_agent = LlmAgent(name="b_node", model=get_model(), description="d", instruction="i")
print("\n建構當下 mode =", node_agent.mode)
wf = Workflow(name="wf", edges=[(START, node_agent)])
resolved = wf.graph.nodes if hasattr(wf, "graph") else None
print("放進 Workflow 之後 mode =", node_agent.mode,
      " ← None 代表要等真的執行時才定案為 single_turn")
print("\n→ 規則：有 parent 就 chat，獨立節點就 single_turn。")

建構當下 mode = None
掛成 sub_agent 之後 mode = chat   ← 交棒得靠 chat

建構當下 mode = None
放進 Workflow 之後 mode = None  ← None 代表要等真的執行時才定案為 single_turn

→ 規則：有 parent 就 chat，獨立節點就 single_turn。


## 7. 📌 補充：context 隔離就是「誰看得到誰的對話」

`chat` 與 `single_turn` 的差別不只是控制權，還有**看得到什麼**。

- `chat`：交棒後 worker 接手同一段對話，**看得到完整歷史**
- `single_turn` / `task`：以工具身分被呼叫，**只拿得到 parent 餵給它的那一句**

這就是為什麼第 4 節的 `boss` 有辦法自己回答「今天星期幾」——
worker 根本沒看到那個問題，它只被問了庫存。

In [7]:
QUESTION = "A-100 還有貨嗎？另外幫我記住我的會員編號是 M-777。"


async def what_did_worker_see(mode):
    """讓 worker 把它「實際收到的內容」覆述出來。"""
    echo = LlmAgent(
        name="echo_worker", model=get_model(),
        description="負責查詢商品庫存",
        instruction="把你這一輪收到的請求原封不動覆述一次，格式：我收到的是「...」。不要回答問題。",
    )
    echo.mode = mode
    boss = LlmAgent(
        name="echo_boss", model=get_model(),
        instruction="你是主管，庫存問題交給 echo_worker 處理。",
        sub_agents=[echo],
    )
    r = InMemoryRunner(agent=boss, app_name="day18")
    sid = await new_session(r)
    print(f"--- mode={mode} ---")
    await ask(r, QUESTION, session_id=sid, trace=True)
    print()


await what_did_worker_see("chat")
await what_did_worker_see("single_turn")

print("→ chat 看得到使用者原話（含 M-777）；single_turn 只拿得到 boss 轉述的那一句。")
print("  這就是 context 隔離：以工具身分被呼叫的 sub-agent，看不到完整對話歷史。")

--- mode=chat ---


  🔧 [echo_boss] 呼叫 transfer_to_agent({'agent_name': 'echo_worker'})
  ↩️  [echo_boss] transfer_to_agent 回傳 {'result': None}


  💬 [echo_worker] 我收到的是「A-100 還有貨嗎？另外幫我記住我的會員編號是 M-777。」

--- mode=single_turn ---


  🔧 [echo_boss] 呼叫 echo_worker({'request': '查詢商品 A-100 的庫存'})


  💬 [echo_worker] 我收到的是「查詢商品 A-100 的庫存」。
  ↩️  [echo_boss] echo_worker 回傳 {'result': '我收到的是「查詢商品 A-100 的庫存」。'}


  💬 [echo_boss] A-100 的庫存狀態我已經請專員幫忙查詢了（請參考上方回覆）。另外，我也已經記住您的會員編號是 **M-777** 了！請問還有其他需要協助的嗎？

→ chat 看得到使用者原話（含 M-777）；single_turn 只拿得到 boss 轉述的那一句。
  這就是 context 隔離：以工具身分被呼叫的 sub-agent，看不到完整對話歷史。


## 8. ⚠️ 補充：「`task` mode 在 graph workflow 停用」——2.8.0 的實測結果

流傳的說法是「`task` mode 自 Python 2.0.0 起在 graph workflow 停用」。
**在 2.8.0 上實測，這個說法不成立**：`LlmAgent` 用 `task` mode 當節點是被允許的。

真正存在的限制是另一條，而且很具體：

In [8]:
# (1) LlmAgent 當節點時，三種 mode 都合法
node_task = LlmAgent(name="node_task", model=get_model(), instruction="i", mode="task")
try:
    wf_task = Workflow(name="wf_task", edges=[(START, node_task)])
    print("✅ LlmAgent(mode='task') 可以當 workflow 節點——沒有被停用")
except Exception as exc:
    print(f"❌ {type(exc).__name__}: {exc}")

# (2) 真正的限制：RemoteA2aAgent 的 task mode 不能當獨立節點
import inspect as _inspect

from google.adk.workflow.utils import _workflow_graph_utils as gu

src = _inspect.getsource(gu)
start = src.find("RemoteA2aAgent in task mode")
print("\n原始碼裡真正寫死的限制：")
print("  " + src[start:start + 120].split("'")[0].strip())

print("\n→ 教材／文章看到「停用」字樣時，"
      "最快的查證方式就是像這樣直接翻你安裝的那個版本的原始碼。")

✅ LlmAgent(mode='task') 可以當 workflow 節點——沒有被停用

原始碼裡真正寫死的限制：
  RemoteA2aAgent in task mode is not supported as a standalone

→ 教材／文章看到「停用」字樣時，最快的查證方式就是像這樣直接翻你安裝的那個版本的原始碼。


## 9. 📌 補充：Managed Agents 是 Preview，而且限制很硬

文章提到 Managed Agents 時只說「Preview」。Preview 的**實際意義**是這些：

In [9]:
import inspect

from google.adk.agents import ManagedAgent

doc = inspect.getdoc(ManagedAgent) or ""
print(doc[:520])
print()
print("mode 欄位只接受:", ManagedAgent.model_fields["mode"].annotation)
print("\n→ 白話：ManagedAgent 只跑得動**伺服器端**工具。"
      "你自己寫的 Python 函式工具（FunctionTool）會被直接拒絕，"
      "所以它現在還取代不了本地的 LlmAgent 團隊。")

An agent backed by the Managed Agents API (interactions.create).

This agent calls the Managed Agents API directly from its execution loop.
Only server-side tools are supported: ADK built-in tools, raw
``google.genai.types.Tool`` configs (the kinds the interactions converter
understands), and server-side remote MCP servers declared as
``RemoteMcpServer`` specs (forwarded to the backend as an ``MCPServerParam``).
Client-executed tools (FunctionTool/callables) and raw
``types.Tool.mcp_servers`` configs are not suppor

mode 欄位只接受: typing.Optional[typing.Literal['single_turn']]

→ 白話：ManagedAgent 只跑得動**伺服器端**工具。你自己寫的 Python 函式工具（FunctionTool）會被直接拒絕，所以它現在還取代不了本地的 LlmAgent 團隊。


## 10. 📌 補充：文章用 LiteLLM 示範多模型——這裡給可跑的替代方案

文章用 LiteLLM 接第三方模型來組異質團隊，但那需要另一把金鑰或本地端點。
**只有一把 Gemini 金鑰也能組異質團隊**：把不同難度的工作分給不同等級的模型。

In [10]:
cheap = LlmAgent(
    name="triage", model=get_model("gemini-flash-lite-latest"),
    description="快速分類客訴屬於哪一類",
    instruction="判斷這則訊息屬於「退費」「技術」「其他」哪一類，只回類別。",
    output_key="kind",
)
smart = LlmAgent(
    name="responder", model=get_model("gemini-flash-latest"),
    instruction="這是一則「{kind?}」類客訴。寫兩句得體的回覆，繁體中文。",
    output_key="reply",
)

from google.adk.agents import SequentialAgent

team = SequentialAgent(name="mixed_team", sub_agents=[cheap, smart])
r4 = InMemoryRunner(agent=team, app_name="day18")
sid4 = await new_session(r4)
await ask(r4, "我三週前申請退費到現在都沒下文，打電話也沒人接。", session_id=sid4)
print_state(await peek_state(r4, sid4))

print("\n→ 便宜的模型做分類、好的模型負責寫字，成本和品質各拿一半。")
print("   真要接非 Gemini 模型時，換成 LiteLlm 即可：")
print("     from google.adk.models.lite_llm import LiteLlm")
print("     LlmAgent(model=LiteLlm(model='openai/gpt-oss-120b', api_base=...))")

  kind: 退費
  reply: 非常抱歉讓您久候多時，對於退款進度延誤以及未能及時接聽您的來電，我們深感歉意。  請您提供當初申請的訂單編號或姓名與電話，我們將立即以專案為您優先查詢並儘速完成退費。

→ 便宜的模型做分類、好的模型負責寫字，成本和品質各拿一半。
   真要接非 Gemini 模型時，換成 LiteLlm 即可：
     from google.adk.models.lite_llm import LiteLlm
     LlmAgent(model=LiteLlm(model='openai/gpt-oss-120b', api_base=...))


## 11. 常見錯誤與踩坑

| 症狀 | 原因 |
|---|---|
| 交棒之後 parent 再也沒講話 | 這就是 `chat` mode 的設計；要收尾請改 `single_turn` |
| 改了 `mode` 卻沒效果 | `mode` 要在**掛上 parent 之前**設定，掛上時才會決定包成哪種工具 |
| `task` mode 的 `output_key` 一直是空的 | task mode 走 `finish_task` 交件，文字回應會被略過 |
| sub-agent 看不到前面的對話 | `single_turn` / `task` 是以工具身分被呼叫，本來就只拿得到那一句 |
| `ManagedAgent` 拒絕你的函式工具 | 它只支援伺服器端工具，這是 Preview 的硬限制 |
| 以為 `task` mode 不能放進 graph | 2.8.0 實測可以；被限制的是 `RemoteA2aAgent` 的 task mode |

## 12. 動手練習

1. 把第 3 節的 `worker.mode` 改成 `single_turn` 再跑一次同樣的問題，
   數數看事件串流裡 `transfer_to_agent` 消失、換成了什麼。
2. 在第 4 節的 `boss` 底下再加一個 `single_turn` 的 `pricing` agent，
   問一個同時需要庫存和報價的問題，確認 boss 會呼叫兩次工具再收尾。
3. 把第 10 節的 `cheap` 和 `smart` 兩個模型對調，
   比較分類品質與最終回覆的差別，估算成本差多少。
4. 幫 `task` mode 的 `booking` agent 接上 runner 跑一次「我要訂位」，
   觀察它怎麼回頭問你人數與日期。

## 本日回顧

- 組團隊的四步驟裡，**第 3 步「決定 mode」最常被跳過，但它決定了對話結構**。
- **`mode` 會改變 parent 實際拿到的工具**：`chat` 什麼都不加、
  `single_turn` 加 `_SingleTurnAgentTool`、`task` 加 `_TaskAgentTool`。
- **`chat` 交棒不回來，`single_turn` / `task` 是工具、會回來。**
  這就是 Day 17「交棒 vs `AgentTool`」的宣告式版本。
- **預設值看你掛在哪**：有 parent 就 `chat`，獨立的 workflow 節點就 `single_turn`。
- ⚠️ **context 隔離**：以工具身分被呼叫的 sub-agent 看不到完整對話歷史。
- ⚠️ **`ManagedAgent` 只支援伺服器端工具**，Preview 的限制是硬的。
- 「task mode 在 graph 被停用」在 2.8.0 **不成立**——遇到這種說法，直接翻原始碼查證。

---
**下一天 → `../day19_a2a_protocol/`**